# 08 — Model Dataset

Join player qualities (pre/post) with team qualities (from projected + to current) into one clean modelling parquet.

**Inputs:**
- `player_qualities_pre_post.parquet` — 7,366 internal transfers, 20 pre + 20 post player qualities
- `team_stats_z_scores_qualities.parquet` — 908 team-season rows, 7 current + 7 projected team qualities

**Output columns:**
- IDs & context (player, teams, league, season, position, age, minutes)
- 7 from-team projected qualities (`from_q_proj_*`)
- 7 to-team current qualities (`to_q_*`)
- 20 pre-transfer player qualities (`pre_*`)
- 20 post-transfer player qualities (`post_*`)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
V1 = f'{DATA_ROOT}/processed_data/model_dataset/v_1'

players = pd.read_parquet(f'{V1}/player_qualities_pre_post.parquet')
teams = pd.read_parquet(f'{V1}/team_stats_z_scores_qualities.parquet')

print(f'Players: {players.shape}')
print(f'Teams:   {teams.shape}')

## 1. Prepare Team Quality Subsets

In [ ]:
# From-team: projected qualities (q_proj_*)
q_proj_cols = [c for c in teams.columns if c.startswith('q_proj_')]
from_team = teams[['team_id', 'competition_id', 'season'] + q_proj_cols].copy()
from_team = from_team.rename(columns={c: f'from_{c}' for c in q_proj_cols})
from_team = from_team.rename(columns={'team_id': 'from_team_id', 'season': 'from_season'})

# To-team: current qualities (q_*)
q_curr_cols = [c for c in teams.columns if c.startswith('q_') and not c.startswith('q_proj_')]
to_team = teams[['team_id', 'competition_id', 'season'] + q_curr_cols].copy()
to_team = to_team.rename(columns={c: f'to_{c}' for c in q_curr_cols})
to_team = to_team.rename(columns={'team_id': 'to_team_id', 'season': 'to_season'})

print(f'From-team columns: {from_team.columns.tolist()}')
print(f'To-team columns:   {to_team.columns.tolist()}')
print(f'\nFrom-team rows: {len(from_team)}')
print(f'To-team rows:   {len(to_team)}')

## 2. Join

In [ ]:
n0 = len(players)

# Join from-team projected qualities
df = players.merge(
    from_team,
    on=['from_team_id', 'competition_id', 'from_season'],
    how='left',
)
n1 = len(df)

# Join to-team current qualities
df = df.merge(
    to_team,
    on=['to_team_id', 'competition_id', 'to_season'],
    how='left',
)
n2 = len(df)

print(f'Before joins:      {n0:,}')
print(f'After from-team:   {n1:,}  (should be same)')
print(f'After to-team:     {n2:,}  (should be same)')

# Check for duplicates from join
if n2 != n0:
    print(f'WARNING: Row count changed! Possible duplicate team-season keys.')

## 3. Check Join Coverage

In [ ]:
from_proj_cols = [c for c in df.columns if c.startswith('from_q_proj_')]
to_curr_cols = [c for c in df.columns if c.startswith('to_q_')]

from_nulls = df[from_proj_cols].isna().any(axis=1).sum()
to_nulls = df[to_curr_cols].isna().any(axis=1).sum()

print(f'Rows missing from-team projected qualities: {from_nulls} ({from_nulls/len(df):.1%})')
print(f'Rows missing to-team current qualities:     {to_nulls} ({to_nulls/len(df):.1%})')

if from_nulls > 0:
    print(f'\nFrom-team null breakdown by season:')
    null_mask = df[from_proj_cols].isna().any(axis=1)
    print(df[null_mask].groupby(['from_season', 'to_season']).size().to_string())

if to_nulls > 0:
    print(f'\nTo-team null breakdown by season:')
    null_mask = df[to_curr_cols].isna().any(axis=1)
    print(df[null_mask].groupby(['from_season', 'to_season']).size().to_string())

## 4. Select & Order Final Columns

In [ ]:
PLAYER_QUALITIES = [
    'Active defence', 'Aerial threat', 'Box threat', 'Chance prevention',
    'Composure', 'Defensive heading', 'Dribbling', 'Effectiveness',
    'Finishing', 'Hold-up play', 'Intelligent defence', 'Involvement',
    'Passing quality', 'Poaching', 'Pressing', 'Progression',
    'Providing teammates', 'Run quality', 'Territorial dominance', 'Winning duels',
]

TEAM_QUALITIES = ['DEFENCE', 'DEFENSIVE_TRANSITION', 'ATTACKING_TRANSITION',
                  'ATTACK', 'PENETRATION', 'CHANCE_CREATION', 'OUTCOME']

# Column order
id_cols = [
    'wy_player_id',
    'from_team_id', 'to_team_id',
    'competition_id', 'league_name',
    'from_season', 'to_season',
    'position',
    'player_season_age',
    'pre_minutes', 'post_minutes',
]

from_team_cols = [f'from_q_proj_{q}' for q in TEAM_QUALITIES]
to_team_cols = [f'to_q_{q}' for q in TEAM_QUALITIES]
pre_player_cols = [f'pre_{q}' for q in PLAYER_QUALITIES]
post_player_cols = [f'post_{q}' for q in PLAYER_QUALITIES]

# Compute deltas
# Team: to_current - from_projected
delta_team_cols = []
for q in TEAM_QUALITIES:
    col = f'delta_team_{q}'
    df[col] = df[f'to_q_{q}'] - df[f'from_q_proj_{q}']
    delta_team_cols.append(col)

# Player: post - pre
delta_player_cols = []
for q in PLAYER_QUALITIES:
    col = f'delta_{q}'
    df[col] = df[f'post_{q}'] - df[f'pre_{q}']
    delta_player_cols.append(col)

all_cols = (id_cols + from_team_cols + to_team_cols + delta_team_cols
            + pre_player_cols + post_player_cols + delta_player_cols)

# Verify
missing = [c for c in all_cols if c not in df.columns]
if missing:
    print(f'WARNING: Missing columns: {missing}')
else:
    print(f'All {len(all_cols)} columns present')

model_df = df[all_cols].reset_index(drop=True)
print(f'\nFinal shape: {model_df.shape}')
print(f'\nColumn groups:')
print(f'  IDs & context:              {len(id_cols)}')
print(f'  From-team projected quals:  {len(from_team_cols)}')
print(f'  To-team current quals:      {len(to_team_cols)}')
print(f'  Delta team quals:           {len(delta_team_cols)}')
print(f'  Pre-transfer player quals:  {len(pre_player_cols)}')
print(f'  Post-transfer player quals: {len(post_player_cols)}')
print(f'  Delta player quals:         {len(delta_player_cols)}')

## 5. Summary Statistics

In [ ]:
print('=== TEAM QUALITIES ===')
print('\nFrom-team projected:')
print(model_df[from_team_cols].describe().round(3).to_string())
print('\nTo-team current:')
print(model_df[to_team_cols].describe().round(3).to_string())
print('\nDelta team (to_current - from_projected):')
print(model_df[delta_team_cols].describe().round(3).to_string())

print('\n=== PLAYER QUALITIES ===')
print(f'{"Quality":25s} {"Pre null%":>10s} {"Post null%":>10s} {"Delta mean":>10s} {"Delta std":>10s}')
print('-' * 70)
for q in PLAYER_QUALITIES:
    pre_n = model_df[f'pre_{q}'].isna().mean()
    post_n = model_df[f'post_{q}'].isna().mean()
    d_mean = model_df[f'delta_{q}'].mean()
    d_std = model_df[f'delta_{q}'].std()
    print(f'{q:25s} {pre_n:>9.1%} {post_n:>10.1%} {d_mean:>+10.3f} {d_std:>10.3f}')

## 6. Save

In [ ]:
out_path = f'{V1}/model_df.parquet'
model_df.to_parquet(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Shape: {model_df.shape}')
print(f'Size: {model_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

print(f'\n=== SUMMARY ===')
print(f'Transfers: {len(model_df):,}')
print(f'Unique players: {model_df["wy_player_id"].nunique():,}')
print(f'Leagues: {model_df["league_name"].nunique()}')
print(f'Windows: {sorted(model_df["to_season"].unique())}')
print(f'Positions: {sorted(model_df["position"].unique())}')
n_team = len(from_team_cols) + len(to_team_cols) + len(delta_team_cols)
n_player = len(pre_player_cols) + len(post_player_cols) + len(delta_player_cols)
print(f'Columns: {model_df.shape[1]} (IDs: {len(id_cols)}, team: {n_team}, player: {n_player})')